#  Baseline Comparison
Compares the LLM's visualisation selection against a rule based baselines across the 20 evaluation episodes. the rule based selection is a deterministic application of the same framework rules as the LLM prompt.




## 1 · Imports & Load Data

In [2]:
import pandas as pd
import numpy as np
import json
from collections import Counter
from IPython.display import display
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import numpy as np


ep = pd.read_csv("episode_features.csv")
df = pd.read_csv("dataset.csv")

with open("evaluation_episodes.json") as f:
    eval_eps = json.load(f)

#Teacher presence helper 
def has_teacher(session, ep_id):
    sub = df[(df["session"] == session) & (df["ep"] == ep_id)]
    return sub["speaker"].dropna().apply(lambda x: str(x).startswith("T0")).any()

# Add teacher flag to eval episodes
for e in eval_eps:
    r = ep[(ep["session"]==e["session"]) & (ep["ep"]==e["ep"])].iloc[0]
    e["features"] = r.to_dict()
    e["has_teacher"] = has_teacher(e["session"], e["ep"])

print(f"Loaded {len(eval_eps)} evaluation episodes")
print(f"Expected viz distribution:")
print(Counter(e["viz"] for e in eval_eps))


Loaded 20 evaluation episodes
Expected viz distribution:
Counter({'Timeline': 4, 'Participation Chart': 4, 'Stacked Bar': 4, 'Network Graph': 4, 'Heatmap': 4})


## 2 · Rule-Based Selector

In [3]:
def rule_based_select(r, has_teacher):
    """
    Deterministic visualisation selection applying the same framework
    rules used in the LLM prompt — in strict priority order.

    Parameters:
    r: dict (episode feature row) 
    has_teacher: bool

    Returns:
    str — selected visualisation name
    """
    gini= r["gini_coefficient"]
    chal= r["challenge_rate"]
    reg= r["regulation_rate"]
    dur= r["total_duration_sec"]
    n_speakers= r["n_speakers"]
    n_turns= r["n_turns"]

    # Priority 1 — Participation Chart
    if gini > 0.6 and has_teacher:
        return "Participation Chart"

    # Priority 2 — Stacked Bar
    if chal > 0.1 and reg > 0.1:
        return "Stacked Bar"

    # Priority 3 — Timeline
    if reg > 0.8 and chal == 0.0 and dur > 120:
        return "Timeline"

    # Priority 4 — Heatmap
    if n_speakers >= 3 and n_turns >= 8 and gini < 0.6:
        return "Heatmap"

    # Priority 5 — Network Graph
    if n_speakers >= 3 and n_turns >= 8:
        return "Network Graph"

    # Default
    return "Timeline"

# Test on eval episodes
print("Rule-based selections:")
print("-" * 55)
for e in eval_eps:
    rb = rule_based_select(e["features"], e["has_teacher"])
    match = "Y" if rb == e["viz"] else "N"
    print(f"  {match} S{e['session']:2d} Ep{e['ep']:2d} | "
          f"Expected: {e['viz']:<22} Rule: {rb}")


Rule-based selections:
-------------------------------------------------------
  Y S22 Ep20 | Expected: Timeline               Rule: Timeline
  Y S15 Ep 5 | Expected: Timeline               Rule: Timeline
  Y S11 Ep15 | Expected: Timeline               Rule: Timeline
  Y S22 Ep 2 | Expected: Timeline               Rule: Timeline
  Y S 8 Ep23 | Expected: Participation Chart    Rule: Participation Chart
  N S15 Ep 6 | Expected: Participation Chart    Rule: Network Graph
  N S 9 Ep17 | Expected: Participation Chart    Rule: Heatmap
  Y S17 Ep 4 | Expected: Participation Chart    Rule: Participation Chart
  Y S22 Ep31 | Expected: Stacked Bar            Rule: Stacked Bar
  Y S 1 Ep 8 | Expected: Stacked Bar            Rule: Stacked Bar
  Y S16 Ep32 | Expected: Stacked Bar            Rule: Stacked Bar
  Y S 5 Ep 7 | Expected: Stacked Bar            Rule: Stacked Bar
  N S13 Ep38 | Expected: Network Graph          Rule: Heatmap
  N S16 Ep17 | Expected: Network Graph          Rule: Heatmap
  N

## 3. Calculate flessis kappa from evaluatores 

In [ ]:
# Flessis kappa based on llm output 
# Rows = items, Columns = [rater1, rater2, rater3, rater4]
# Values must be 0, 1, or 2
data_cat1 = np.array([
    [2, 2, 2],  # item 1
    [2, 2, 2],  # item 2
    [1, 2, 2],
    [1, 2, 2],
    [2, 2, 1],
    [1, 1, 1],
    [2, 2, 2],
    [2, 2, 1],
    [1, 1, 1],
    [2, 2, 2],
    [2, 2, 2],
    [1, 1, 1],
    [1, 1, 1],
    [2, 2, 2],
    [2, 2, 2],
    [2, 2, 2],
    [1, 1, 1],
    [1, 1, 1],
    [2, 2, 2],
    [2, 2, 2],
])

data_cat2 = np.array([
    [1, 1, 1],
    [1, 1, 1],
    [2, 1, 1],
    [2, 2, 2],
    [2, 1, 1],
    [1, 1, 1],
    [2, 2, 2],
    [1, 1, 1],
    [1, 1, 1],
    [1, 1, 1],
    [1, 1, 2],
    [2, 2, 2],
    [1, 1, 1],
    [2, 2, 2],
    [1, 2, 1],
    [1, 1, 1],
    [1, 1, 1],
    [1, 1, 1],
    [1, 2, 1],
    [2, 2, 2],
])

data_cat3 = np.array([
    [1, 1, 1],
    [1, 1, 2],
    [2, 1, 1],
    [2, 2, 2],
    [2, 2, 2],
    [2, 2, 2],
    [2, 2, 1],
    [2, 2, 2],
    [1, 1, 1],
    [2, 2, 2],
    [1, 1, 1],
    [1, 1, 1],
    [2, 1, 1],
    [2, 2, 2],
    [1, 1, 1],
    [1, 1, 1],
    [2, 2, 2],
    [1, 1, 1],
    [2, 2, 2],
    [2, 2, 2],
    
])

for i, data in enumerate([data_cat1, data_cat2, data_cat3], 1):
    table, _ = aggregate_raters(data)
    kappa = fleiss_kappa(table)
    print(f"Category {i} κ: {kappa:.3f}")

Category 1 κ: 0.713
Category 2 κ: 0.625
Category 3 κ: 0.732
